# 질의 변형 
- 질의 변형은 사용자의 원래 질문을 보다 효과적인 검색 쿼리로 변환하는 과정을 말한다. 
- 기본적인 RAG 검색 시스템에서는 사용자 질문을 그대로 사용하여 관련 문서를 검색한다.
- 사용자의 질문이 모호하거나 질문에 최적화되지 않을 경우 연관 문서 검색이 제대로 되지 않는다. 

## 다중 질의 생성 
- 질의 변형의 한 기법으로 사용자의 원래 질문을 바탕으로 여러 개의 다양ㅇ한 쿼리를 생성하는 방법이다. 
- 다중질의 생성 -> 병렬 검색 -> 결과 통합 

In [1]:
import logging 

logging.basicConfig()
logging.getLogger("langchain.retrievers.multiquery").setLevel(logging.INFO)

In [4]:
from langchain_community.vectorstores import Chroma 
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 

loaders = [TextLoader("./data/How_to_invest_money.txt")]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [ ]:
recursive_spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
)

split_docs = recursive_spliter.split_documents(docs)


In [48]:
import chromadb

embeddings = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    tenant="default_tenant",
    database="default_database",
)

print(client.list_collections())

try:
    client.delete_collection("multi_query")
except Exception:
    pass 


vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="multi_query",
    client=client
)

[Collection(name=multi_modal_rag), Collection(name=split_parents), Collection(name=multi_query)]


In [49]:
print(vectorstore._collection)

Collection(name=multi_query)


In [50]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI 


In [51]:
from langchain_openai import ChatOpenAI

def get_model(model : str = "gemma-3-4b-it"):
    return ChatOpenAI(
        model=model,
        temperature=0.2, 
        base_url="http://localhost:1234/v1",
        api_key="lm-studio"
)

In [52]:
def get_multiquery_retriever(llm):
    return MultiQueryRetriever.from_llm(
        retriever=vectorstore.as_retriever(),
        llm = llm, 
    )

In [53]:
llm = get_model("llama-3.2-Korean-Bllossom-3B-GGUF")
retriever =get_multiquery_retriever(llm)

unique_docs = retriever.invoke("주식 투자를 처음 할려면 어떻게 해야 하나요?")

In [54]:
print(len(unique_docs))

7


In [56]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

question = "주식 투자를 처음 할려면 어떻게 해야 하나요?"

docs = retriever.invoke(question)

# print(len(docs))
# print(len(docs[0].page_content))
prompt = ChatPromptTemplate.from_template("""
다음 문서를 참고해서 한국어로 답변하세요.
문서가 영어여도 답변은 반드시 한국어로 하세요.

문서:
{context}

질문:
{question}
""")

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "context": format_docs(docs),
    "question": question
})

print(answer)


제공해주신 문서는 투자에 대한 고전적이고 심도 있는 원칙들을 담고 있습니다. 이 글은 주식 투자를 처음 시작하는 초보자에게 "어떤 종목을 사라"는 구체적인 지침보다는, **성공적인 투자를 하기 위해 갖춰야 할 근본적인 자세와 지식**을 강조하고 있습니다.

따라서 이 문서를 바탕으로 주식 투자를 처음 시작하려는 분께 드릴 수 있는 조언은 다음과 같습니다.

---

### 📚 성공적인 투자를 위한 근본 원칙 (The Foundational Principles)

이 글의 저자는 투자에 뛰어들기 전에 반드시 갖춰야 할 지식과 태도가 있다고 강조합니다.

**1. 자신의 요구 사항을 명확히 하십시오 (Define Your Requirements)**
*   가장 중요한 단계입니다. 투자를 시작하기 전에 "나는 무엇을 원하는가?"에 대한 답을 스스로 내려야 합니다.
*   **질문:** 나의 투자 목표는 무엇인가요? (단순히 돈을 버는 것인지, 안정적인 현금 흐름을 만드는 것인지 등)
*   **경고:** 목표 없이 은행가에게 종목 추천을 받으러 가는 것은, 증상도 말하지 않은 채 의사에게 약을 달라고 하는 것만큼 어리석은 행동이라고 저자는 경고합니다.

**2. 투자의 다섯 가지 핵심 요소를 이해하십시오 (Understand the Five Selection Criteria)**
모든 투자 대상을 선택할 때 고려해야 할 다섯 가지 중요한 기준이 있습니다. 이 원칙을 숙지하는 것이 중요합니다.
*   **원금 및 이자의 안전성 (Safety of principal and interest):** 투자 원금과 약속된 이자를 받을 수 있다는 확신이 있습니까?
*   **수익률 (Rate of income):** 실제로 투자한 금액 대비 순이익(net return)은 얼마입니까?
*   **현금화 용이성 (Convertibility into cash):** 필요할 때 투자금을 얼마나 빠르고 쉽게 현금으로 바꿀 수 있습니까?
*   **가치 상승 가능성 (Prospect o

In [59]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm = get_model(),
    chain_type = "stuff", 
    retriever = retriever, 
    return_source_documents=True,
)

result = qa_chain.invoke({"query" : question}) 

print(f"답변 {result["result"]}")
print("사용된 문서")
for doc in result["source_documents"]:
    print(doc.page_content)

답변 제공해주신 자료는 1908년에 쓰인 투자 서적의 일부로, 당시의 투자 원칙을 담고 있습니다. 따라서 현대적인 주식 시장 상황과는 차이가 있을 수 있으나, 이 책에서 강조하는 **근본적인 투자 원칙**을 바탕으로 주식 투자를 시작할 때 필요한 자세에 대해 조언해 드릴 수 있습니다.

이 책의 저자는 주식 투자를 포함한 모든 투자에 있어 가장 중요한 것은 **'준비와 이해'**라고 강조합니다.

다음은 본문에서 도출할 수 있는, 주식 투자를 처음 시작하는 분께 드리는 핵심 원칙들입니다.

### 1. 투자 목표를 명확히 하십시오 (가장 중요)
*   **본문의 가르침:** "투자 시도를 하기 전에 자신의 진정한 요구 사항에 대해 철저히 이해해야 한다."
*   **적용:** 주식 투자를 시작하기 전에, **"내가 이 돈을 왜 투자하는가?", "언제 필요한가?", "얼마의 위험까지 감수할 수 있는가?"**에 대한 답을 스스로 내려야 합니다. 목표 없이 뛰어드는 것은 의사에게 증상도 말하지 않은 채 약만 달라고 하는 것만큼 어리석다는 것이 저자의 비유입니다.

### 2. 투자 상품에 대한 지식을 쌓으십시오
*   **본문의 가르침:** "각 투자 형태의 근본적인 차이점들을 명확히 이해해야 한다."
*   **적용:** 주식(Stocks)이라는 것이 무엇인지, 그것이 기업의 어떤 부분에 투자하는 것인지 근본적인 지식을 습득해야 합니다. 단순히 '오른다/내린다'는 소문에 의존해서는 안 됩니다.

### 3. 시장의 흐름을 이해하십시오
*   **본문의 가르침:** "증권 가격 변동을 통제하는 일반적인 원칙들을 확실히 파악해야 한다."
*   **적용:** 주식 시장의 단기적인 등락에 일희일비하기보다는, **'왜 이런 큰 가격 변동이 발생하는가?'**라는 거시적인 흐름과 원인을 파악하는 것이 중요합니다. 시장의 큰 추세를 읽을 수 있다면, 투자금을 효과적으로 늘릴 기회를 잡을 수 있습니다.

---
**요약하자면, 이 책이 주는 가장 강력한 메시지는 다음과 같습니다